# AcademicMate · Week 3 data pipeline

Copy of the shared, PDF and image code in the Topic 3 lab. Run Cell 1 in Colab, then upload the five PDFs and two PNGs from the raw folder to **/content/week3_project/data/raw** using the Files panel. Keep them at the same level. Run the remaining cells in order.

The PDF blocks are repeated for the five files because the handout reads one PDF at a time. Only input paths, per-file output names and the chosen screenshot size change. Each code cell records its handout page and its exact edits in notebook metadata. The text file stores the source filename. The handout leaves the page number empty. Its image code saves RGB PNGs without reading their text.

The saved checks used local copies of the inputs and a local workspace in place of /content/week3_project. Full PDF text previews were cleared from this copy. Running privately in Colab will show the original previews.

## Part A · Set up and check the files

Cells 1 and 3, pages 4 and 6. Pause after Cell 1 to upload the seven files. Cell 2 is for the optional dummy ZIP.

In [1]:
# Cell 1 — Workspace setup (RUN FIRST)
from pathlib import Path

BASE_DIR = Path("/content/week3_project")
RAW_DIR = BASE_DIR / "data" / "raw"
PROCESSED_DIR = BASE_DIR / "data" / "processed"

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Compatibility aliases for notebook examples
RAW = RAW_DIR
PROCESSED = PROCESSED_DIR
BASE = BASE_DIR
OUT = PROCESSED_DIR

print("Raw folder:", RAW_DIR)
print("Processed folder:", PROCESSED_DIR)

Raw folder: C:\vscode\oya\ai-engineering-utm-academicmate\data\raw
Processed folder: C:\vscode\oya\ai-engineering-utm-academicmate\data\processed


In [2]:
# Cell 3 — Verify uploaded inputs (self-contained)
from pathlib import Path
RAW = Path("/content/week3_project/data/raw")
RAW_DIR = RAW
BASE_DIR = RAW.parent.parent
PROCESSED_DIR = BASE_DIR / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
BASE = BASE_DIR
OUT = PROCESSED_DIR

files = sorted(f for f in RAW.rglob("*") if f.is_file())
print("Input count:", len(files))
for f in files:
  print(f.relative_to(RAW), "bytes:", f.stat().st_size)

assert files, "Upload at least one relevant input file before proceeding."
print("Upload verified. Select your relevant input(s) for Step A3.")

Input count: 7
Activity for Module 2.pdf bytes: 23576
Assignment 1[Individual].pdf bytes: 277490
NLP Final Project Question.pdf bytes: 3360535
Post Module Assessment_SAIA3453_final.pdf bytes: 257513
SAIA 2163 — Assignment 2_ Text Vectorization & Embedding Methods.pdf bytes: 3002276
Screenshot 2026-11-02 093501.png bytes: 63628
Screenshot 2026-11-02 094157.png bytes: 55883
Upload verified. Select your relevant input(s) for Step A3.


## Part B · Record where each result came from

Cells 4 and 4A, pages 12–15.

In [3]:
# Cell 4 — REQUIRED: Source Selection + Lineage Utilities
# Run ONCE after Cell 1 and Cell 3

from pathlib import Path
import json
import hashlib
from datetime import datetime, timezone

# Select the exact source before parsing.
# Never automatically take the first file by extension.

def select_source(relative_path=None, filename=None, suffixes=None):
    if relative_path:
        selected = RAW_DIR / relative_path
        if not selected.is_file():
            raise FileNotFoundError(
                f"Source not found: {selected}"
            )
        return selected
    matches = sorted(
        p for p in RAW_DIR.rglob("*")
        if p.is_file()
        and (filename is None or p.name == filename)
        and (suffixes is None or p.suffix.lower() in suffixes)
    )
    print(
        "Candidates:",
        [str(p.relative_to(RAW_DIR)) for p in matches]
    )
    if len(matches) != 1:
        raise ValueError(
            "Select exactly one source. "
            "Set relative_path to an exact file under RAW_DIR."
        )
    return matches[0]
def source_hash(path):
    return hashlib.sha256(
        Path(path).read_bytes()
    ).hexdigest()
def save_manifest(records):
    manifest = {
        "pipeline_version": "week3-v1",
        "generated_at_utc": datetime.now(timezone.utc).isoformat(),
        "records": records
    }
    target = BASE_DIR / "data" / "week3_manifest.json"
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(
        json.dumps(
            manifest,
            indent=2,
            ensure_ascii=False
        ),
        encoding="utf-8"
    )
    print("Saved lineage:", target)
    return manifest

# Confirmation messages
print("Cell 4 completed successfully.")
print("Source selection and lineage utilities are ready.")
print("Available functions:")
print("- select_source()")
print("- source_hash()")
print("- save_manifest()")

Cell 4 completed successfully.
Source selection and lineage utilities are ready.
Available functions:
- select_source()
- source_hash()
- save_manifest()


In [4]:
# CELL 4A — COMMON LINEAGE REGISTRY
# REQUIRED: Run once before selected Part B pathways
from pathlib import Path
lineage_entries = {}
def register_lineage(source, output, processing):
    source = Path(source).resolve()
    output = Path(output).resolve()
    # Validate source
    if not source.is_file():
        raise FileNotFoundError(
            f"Source file not found: {source}"
        )
    # Validate prepared output
    if not output.is_file():
        raise FileNotFoundError(
            f"Prepared output not found: {output}"
        )
    if output.stat().st_size == 0:
        raise ValueError(
            f"Prepared output is empty: {output}"
        )
    # Validate processing description
    if not isinstance(processing, str) or not processing.strip():
        raise ValueError(
            "A processing description is required."
        )
    processing = processing.strip()
    if processing.upper().startswith("REPLACE"):
        raise ValueError(
            "Replace the placeholder with the actual method."
        )
    # Ensure source and output belong to project workspace
    source.relative_to(RAW_DIR.resolve())
    output.relative_to(PROCESSED_DIR.resolve())
    # Same source/output pair is updated, not duplicated
    key = (str(source), str(output))
    lineage_entries[key] = {
        "source": source,
        "output": output,
        "processing": processing
    }
    print(
        "Registered:",
        source.name,
        "->",
        output.name
    )
print("Common lineage registry ready.")
print("Registered entries:", len(lineage_entries))

Common lineage registry ready.
Registered entries: 0


### S01 · Activity for Module 2.pdf

Cells 5A1 and 5A2, pages 16–20. Compare fixed and paragraph-aware chunks. Then check and register this PDF on pages 35–36.

In [5]:
# Cell 5A1
# PATHWAY A1 — PDF/TXT
# CONDITIONAL REQUIRED: Run only for document-based projects
# Install PDF processing library
!pip -q install pypdf

from pypdf import PdfReader
import re

# Step 1: Select the exact source file
# Example: Synthetic practice PDF
# Replace with your actual project file where applicable.
f = select_source(
    relative_path=(
        "Activity for Module 2.pdf"
    )
)
# Step 2: Extract text according to file type
if f.suffix.lower() == ".pdf":
    reader = PdfReader(str(f))
    pages = [
        (i + 1, page.extract_text() or "")
        for i, page in enumerate(reader.pages)
    ]
elif f.suffix.lower() == ".txt":
    pages = [(1, f.read_text(encoding="utf-8"))]
else:
    raise ValueError("Pathway A needs a PDF or TXT source")
# Step 3: Basic text cleaning
text = "\n".join(content for _, content in pages)
clean = re.sub(r"[ \t]+", " ", text).strip()

print("Source:", f.name)
print("Pages/segments:", len(pages))
print("Extracted characters:", len(text))
print("Cleaned characters:", len(clean))
print("Content sample:", repr(clean[:300]))

assert clean, "No usable text: check extraction or OCR needs."
# Step 4: Fixed-size chunking
def fixed_chunks(s, size=500):
    if size <= 0:
        raise ValueError("Chunk size must be positive")
    return [
        s[i:i + size]
        for i in range(0, len(s), size)
    ]
# Step 5: Paragraph-aware chunking
def paragraph_chunks(s, size=500):
    if size <= 0:
        raise ValueError("Chunk size must be positive")
    parts = []
    current = ""
    for segment in re.split(r"\n\s*\n", s):
        segment = segment.strip()
        if not segment:
            continue
        if len(segment) > size:
            if current:
                parts.append(current)
                current = ""
            parts.extend(fixed_chunks(segment, size))

        elif len(current) + len(segment) + (2 if current else 0) <= size:
            current += ("\n\n" if current else "") + segment
        else:
            if current:
                parts.append(current)
            current = segment
    if current:
        parts.append(current)
    return parts
# Step 6: Compare two preparation approaches
fixed = fixed_chunks(clean, size=500)
structured = paragraph_chunks(clean, size=500)

print("\n--- Chunking Comparison ---")
print("Fixed chunks:", len(fixed))
print("Paragraph-aware chunks:", len(structured))

print("\nFixed sample:")
print(repr(fixed[0][:180]))

print("\nParagraph-aware sample:")
print(repr(structured[0][:180]))

print("\nPATHWAY A1 completed successfully.")

Source: Activity for Module 2.pdf
Pages/segments: 1
Extracted characters: 600
Cleaned characters: 600
Fixed chunks: 2
Paragraph-aware chunks: 2
PATHWAY A1 completed successfully.


In [6]:
# Cell 5A2
# PATHWAY A2 — REQUIRED after A1
# Choose and export traceable text chunks
import json
# Step 1: Select the preparation approach
# Change to "fixed" if your Part A1 evidence supports it.
CHOSEN_METHOD = "paragraph-aware"
if CHOSEN_METHOD == "paragraph-aware":
    chosen = structured
    approach = "paragraph-aware-char-500"
elif CHOSEN_METHOD == "fixed":
    chosen = fixed
    approach = "fixed-char-500"
else:
    raise ValueError("Choose 'paragraph-aware' or 'fixed'")
# Step 2: Create traceable records
records = [
    {
        "record_id": f"{f.stem}-{i:04d}",
        "text": chunk,
        "source": str(f.relative_to(RAW_DIR)),
        "source_sha256": source_hash(f),
        "processing": approach,
        "page": None
    }
    for i, chunk in enumerate(chosen, 1)
]
# Page is None because the basic whole-document
# splitter does not preserve page boundaries.
# Step 3: Export prepared records
prepared_path = PROCESSED_DIR / "S01" / "prepared_text.jsonl"
prepared_path.parent.mkdir(parents=True, exist_ok=True)
with prepared_path.open("w", encoding="utf-8") as output:
    for row in records:
        output.write(
            json.dumps(row, ensure_ascii=False) + "\n"
        )
# Step 4: Basic validation
assert prepared_path.is_file(), "Prepared output not found"
assert prepared_path.stat().st_size > 0, "Prepared output is empty"
assert records, "No text chunks were produced"
assert all(row["text"].strip() for row in records), \
    "One or more chunks are empty"
with prepared_path.open("r", encoding="utf-8") as file:
    exported_records = [
        json.loads(line)
        for line in file
        if line.strip()
    ]
assert len(exported_records) == len(records), \
    "Exported record count does not match"

# Step 5: Display evidence
print("PATHWAY A2 completed successfully.")
print("Selected approach:", approach)
print("Total chunks exported:", len(records))
print("Prepared file:", prepared_path)
print("File size:", prepared_path.stat().st_size, "bytes")
print("Validation: PASS")
print("First record:", records[0])

PATHWAY A2 completed successfully.
Selected approach: paragraph-aware-char-500
Total chunks exported: 2
Prepared file: C:\vscode\oya\ai-engineering-utm-academicmate\data\processed\S01\prepared_text.jsonl
File size: 1086 bytes
Validation: PASS


In [7]:
assert prepared_path.exists(), 'Prepared output missing.'
assert prepared_path.stat().st_size > 0, 'Prepared output is empty.'
print('Output:',prepared_path, 'Size:',prepared_path.stat().st_size)
print('Validation: file exists and is non-empty; verify content manually too.')

Output: C:\vscode\oya\ai-engineering-utm-academicmate\data\processed\S01\prepared_text.jsonl Size: 1086
Validation: file exists and is non-empty; verify content manually too.


In [8]:
# Register Pathway A — PDF/TXT
register_lineage(
    source=f,
    output=prepared_path,
    processing=approach
)

Registered: Activity for Module 2.pdf -> prepared_text.jsonl


### S02 · Assignment 1[Individual].pdf

Cells 5A1 and 5A2, pages 16–20. Compare fixed and paragraph-aware chunks. Then check and register this PDF on pages 35–36.

In [9]:
# Cell 5A1
# PATHWAY A1 — PDF/TXT
# CONDITIONAL REQUIRED: Run only for document-based projects
# Install PDF processing library
!pip -q install pypdf

from pypdf import PdfReader
import re

# Step 1: Select the exact source file
# Example: Synthetic practice PDF
# Replace with your actual project file where applicable.
f = select_source(
    relative_path=(
        "Assignment 1[Individual].pdf"
    )
)
# Step 2: Extract text according to file type
if f.suffix.lower() == ".pdf":
    reader = PdfReader(str(f))
    pages = [
        (i + 1, page.extract_text() or "")
        for i, page in enumerate(reader.pages)
    ]
elif f.suffix.lower() == ".txt":
    pages = [(1, f.read_text(encoding="utf-8"))]
else:
    raise ValueError("Pathway A needs a PDF or TXT source")
# Step 3: Basic text cleaning
text = "\n".join(content for _, content in pages)
clean = re.sub(r"[ \t]+", " ", text).strip()

print("Source:", f.name)
print("Pages/segments:", len(pages))
print("Extracted characters:", len(text))
print("Cleaned characters:", len(clean))
print("Content sample:", repr(clean[:300]))

assert clean, "No usable text: check extraction or OCR needs."
# Step 4: Fixed-size chunking
def fixed_chunks(s, size=500):
    if size <= 0:
        raise ValueError("Chunk size must be positive")
    return [
        s[i:i + size]
        for i in range(0, len(s), size)
    ]
# Step 5: Paragraph-aware chunking
def paragraph_chunks(s, size=500):
    if size <= 0:
        raise ValueError("Chunk size must be positive")
    parts = []
    current = ""
    for segment in re.split(r"\n\s*\n", s):
        segment = segment.strip()
        if not segment:
            continue
        if len(segment) > size:
            if current:
                parts.append(current)
                current = ""
            parts.extend(fixed_chunks(segment, size))

        elif len(current) + len(segment) + (2 if current else 0) <= size:
            current += ("\n\n" if current else "") + segment
        else:
            if current:
                parts.append(current)
            current = segment
    if current:
        parts.append(current)
    return parts
# Step 6: Compare two preparation approaches
fixed = fixed_chunks(clean, size=500)
structured = paragraph_chunks(clean, size=500)

print("\n--- Chunking Comparison ---")
print("Fixed chunks:", len(fixed))
print("Paragraph-aware chunks:", len(structured))

print("\nFixed sample:")
print(repr(fixed[0][:180]))

print("\nParagraph-aware sample:")
print(repr(structured[0][:180]))

print("\nPATHWAY A1 completed successfully.")

Source: Assignment 1[Individual].pdf
Pages/segments: 3
Extracted characters: 4690
Cleaned characters: 4683
Fixed chunks: 10
Paragraph-aware chunks: 11
PATHWAY A1 completed successfully.


In [10]:
# Cell 5A2
# PATHWAY A2 — REQUIRED after A1
# Choose and export traceable text chunks
import json
# Step 1: Select the preparation approach
# Change to "fixed" if your Part A1 evidence supports it.
CHOSEN_METHOD = "paragraph-aware"
if CHOSEN_METHOD == "paragraph-aware":
    chosen = structured
    approach = "paragraph-aware-char-500"
elif CHOSEN_METHOD == "fixed":
    chosen = fixed
    approach = "fixed-char-500"
else:
    raise ValueError("Choose 'paragraph-aware' or 'fixed'")
# Step 2: Create traceable records
records = [
    {
        "record_id": f"{f.stem}-{i:04d}",
        "text": chunk,
        "source": str(f.relative_to(RAW_DIR)),
        "source_sha256": source_hash(f),
        "processing": approach,
        "page": None
    }
    for i, chunk in enumerate(chosen, 1)
]
# Page is None because the basic whole-document
# splitter does not preserve page boundaries.
# Step 3: Export prepared records
prepared_path = PROCESSED_DIR / "prepared_text.jsonl"
prepared_path.parent.mkdir(parents=True, exist_ok=True)
with prepared_path.open("w", encoding="utf-8") as output:
    for row in records:
        output.write(
            json.dumps(row, ensure_ascii=False) + "\n"
        )
# Step 4: Basic validation
assert prepared_path.is_file(), "Prepared output not found"
assert prepared_path.stat().st_size > 0, "Prepared output is empty"
assert records, "No text chunks were produced"
assert all(row["text"].strip() for row in records), \
    "One or more chunks are empty"
with prepared_path.open("r", encoding="utf-8") as file:
    exported_records = [
        json.loads(line)
        for line in file
        if line.strip()
    ]
assert len(exported_records) == len(records), \
    "Exported record count does not match"

# Step 5: Display evidence
print("PATHWAY A2 completed successfully.")
print("Selected approach:", approach)
print("Total chunks exported:", len(records))
print("Prepared file:", prepared_path)
print("File size:", prepared_path.stat().st_size, "bytes")
print("Validation: PASS")
print("First record:", records[0])

PATHWAY A2 completed successfully.
Selected approach: paragraph-aware-char-500
Total chunks exported: 11
Prepared file: C:\vscode\oya\ai-engineering-utm-academicmate\data\processed\prepared_text.jsonl
File size: 7446 bytes
Validation: PASS


In [11]:
assert prepared_path.exists(), 'Prepared output missing.'
assert prepared_path.stat().st_size > 0, 'Prepared output is empty.'
print('Output:',prepared_path, 'Size:',prepared_path.stat().st_size)
print('Validation: file exists and is non-empty; verify content manually too.')

Output: C:\vscode\oya\ai-engineering-utm-academicmate\data\processed\prepared_text.jsonl Size: 7446
Validation: file exists and is non-empty; verify content manually too.


In [12]:
# Register Pathway A — PDF/TXT
register_lineage(
    source=f,
    output=prepared_path,
    processing=approach
)

Registered: Assignment 1[Individual].pdf -> prepared_text.jsonl


### S03 · NLP Final Project Question.pdf

Cells 5A1 and 5A2, pages 16–20. Compare fixed and paragraph-aware chunks. Then check and register this PDF on pages 35–36.

In [13]:
# Cell 5A1
# PATHWAY A1 — PDF/TXT
# CONDITIONAL REQUIRED: Run only for document-based projects
# Install PDF processing library
!pip -q install pypdf

from pypdf import PdfReader
import re

# Step 1: Select the exact source file
# Example: Synthetic practice PDF
# Replace with your actual project file where applicable.
f = select_source(
    relative_path=(
        "NLP Final Project Question.pdf"
    )
)
# Step 2: Extract text according to file type
if f.suffix.lower() == ".pdf":
    reader = PdfReader(str(f))
    pages = [
        (i + 1, page.extract_text() or "")
        for i, page in enumerate(reader.pages)
    ]
elif f.suffix.lower() == ".txt":
    pages = [(1, f.read_text(encoding="utf-8"))]
else:
    raise ValueError("Pathway A needs a PDF or TXT source")
# Step 3: Basic text cleaning
text = "\n".join(content for _, content in pages)
clean = re.sub(r"[ \t]+", " ", text).strip()

print("Source:", f.name)
print("Pages/segments:", len(pages))
print("Extracted characters:", len(text))
print("Cleaned characters:", len(clean))
print("Content sample:", repr(clean[:300]))

assert clean, "No usable text: check extraction or OCR needs."
# Step 4: Fixed-size chunking
def fixed_chunks(s, size=500):
    if size <= 0:
        raise ValueError("Chunk size must be positive")
    return [
        s[i:i + size]
        for i in range(0, len(s), size)
    ]
# Step 5: Paragraph-aware chunking
def paragraph_chunks(s, size=500):
    if size <= 0:
        raise ValueError("Chunk size must be positive")
    parts = []
    current = ""
    for segment in re.split(r"\n\s*\n", s):
        segment = segment.strip()
        if not segment:
            continue
        if len(segment) > size:
            if current:
                parts.append(current)
                current = ""
            parts.extend(fixed_chunks(segment, size))

        elif len(current) + len(segment) + (2 if current else 0) <= size:
            current += ("\n\n" if current else "") + segment
        else:
            if current:
                parts.append(current)
            current = segment
    if current:
        parts.append(current)
    return parts
# Step 6: Compare two preparation approaches
fixed = fixed_chunks(clean, size=500)
structured = paragraph_chunks(clean, size=500)

print("\n--- Chunking Comparison ---")
print("Fixed chunks:", len(fixed))
print("Paragraph-aware chunks:", len(structured))

print("\nFixed sample:")
print(repr(fixed[0][:180]))

print("\nParagraph-aware sample:")
print(repr(structured[0][:180]))

print("\nPATHWAY A1 completed successfully.")

Source: NLP Final Project Question.pdf
Pages/segments: 12
Extracted characters: 16285
Cleaned characters: 16264
Fixed chunks: 33
Paragraph-aware chunks: 33
PATHWAY A1 completed successfully.


In [14]:
# Cell 5A2
# PATHWAY A2 — REQUIRED after A1
# Choose and export traceable text chunks
import json
# Step 1: Select the preparation approach
# Change to "fixed" if your Part A1 evidence supports it.
CHOSEN_METHOD = "paragraph-aware"
if CHOSEN_METHOD == "paragraph-aware":
    chosen = structured
    approach = "paragraph-aware-char-500"
elif CHOSEN_METHOD == "fixed":
    chosen = fixed
    approach = "fixed-char-500"
else:
    raise ValueError("Choose 'paragraph-aware' or 'fixed'")
# Step 2: Create traceable records
records = [
    {
        "record_id": f"{f.stem}-{i:04d}",
        "text": chunk,
        "source": str(f.relative_to(RAW_DIR)),
        "source_sha256": source_hash(f),
        "processing": approach,
        "page": None
    }
    for i, chunk in enumerate(chosen, 1)
]
# Page is None because the basic whole-document
# splitter does not preserve page boundaries.
# Step 3: Export prepared records
prepared_path = PROCESSED_DIR / "S03" / "prepared_text.jsonl"
prepared_path.parent.mkdir(parents=True, exist_ok=True)
with prepared_path.open("w", encoding="utf-8") as output:
    for row in records:
        output.write(
            json.dumps(row, ensure_ascii=False) + "\n"
        )
# Step 4: Basic validation
assert prepared_path.is_file(), "Prepared output not found"
assert prepared_path.stat().st_size > 0, "Prepared output is empty"
assert records, "No text chunks were produced"
assert all(row["text"].strip() for row in records), \
    "One or more chunks are empty"
with prepared_path.open("r", encoding="utf-8") as file:
    exported_records = [
        json.loads(line)
        for line in file
        if line.strip()
    ]
assert len(exported_records) == len(records), \
    "Exported record count does not match"

# Step 5: Display evidence
print("PATHWAY A2 completed successfully.")
print("Selected approach:", approach)
print("Total chunks exported:", len(records))
print("Prepared file:", prepared_path)
print("File size:", prepared_path.stat().st_size, "bytes")
print("Validation: PASS")
print("First record:", records[0])

PATHWAY A2 completed successfully.
Selected approach: paragraph-aware-char-500
Total chunks exported: 33
Prepared file: C:\vscode\oya\ai-engineering-utm-academicmate\data\processed\S03\prepared_text.jsonl
File size: 25169 bytes
Validation: PASS


In [15]:
assert prepared_path.exists(), 'Prepared output missing.'
assert prepared_path.stat().st_size > 0, 'Prepared output is empty.'
print('Output:',prepared_path, 'Size:',prepared_path.stat().st_size)
print('Validation: file exists and is non-empty; verify content manually too.')

Output: C:\vscode\oya\ai-engineering-utm-academicmate\data\processed\S03\prepared_text.jsonl Size: 25169
Validation: file exists and is non-empty; verify content manually too.


In [16]:
# Register Pathway A — PDF/TXT
register_lineage(
    source=f,
    output=prepared_path,
    processing=approach
)

Registered: NLP Final Project Question.pdf -> prepared_text.jsonl


### S04 · Post Module Assessment_SAIA3453_final.pdf

Cells 5A1 and 5A2, pages 16–20. Compare fixed and paragraph-aware chunks. Then check and register this PDF on pages 35–36.

In [17]:
# Cell 5A1
# PATHWAY A1 — PDF/TXT
# CONDITIONAL REQUIRED: Run only for document-based projects
# Install PDF processing library
!pip -q install pypdf

from pypdf import PdfReader
import re

# Step 1: Select the exact source file
# Example: Synthetic practice PDF
# Replace with your actual project file where applicable.
f = select_source(
    relative_path=(
        "Post Module Assessment_SAIA3453_final.pdf"
    )
)
# Step 2: Extract text according to file type
if f.suffix.lower() == ".pdf":
    reader = PdfReader(str(f))
    pages = [
        (i + 1, page.extract_text() or "")
        for i, page in enumerate(reader.pages)
    ]
elif f.suffix.lower() == ".txt":
    pages = [(1, f.read_text(encoding="utf-8"))]
else:
    raise ValueError("Pathway A needs a PDF or TXT source")
# Step 3: Basic text cleaning
text = "\n".join(content for _, content in pages)
clean = re.sub(r"[ \t]+", " ", text).strip()

print("Source:", f.name)
print("Pages/segments:", len(pages))
print("Extracted characters:", len(text))
print("Cleaned characters:", len(clean))
print("Content sample:", repr(clean[:300]))

assert clean, "No usable text: check extraction or OCR needs."
# Step 4: Fixed-size chunking
def fixed_chunks(s, size=500):
    if size <= 0:
        raise ValueError("Chunk size must be positive")
    return [
        s[i:i + size]
        for i in range(0, len(s), size)
    ]
# Step 5: Paragraph-aware chunking
def paragraph_chunks(s, size=500):
    if size <= 0:
        raise ValueError("Chunk size must be positive")
    parts = []
    current = ""
    for segment in re.split(r"\n\s*\n", s):
        segment = segment.strip()
        if not segment:
            continue
        if len(segment) > size:
            if current:
                parts.append(current)
                current = ""
            parts.extend(fixed_chunks(segment, size))

        elif len(current) + len(segment) + (2 if current else 0) <= size:
            current += ("\n\n" if current else "") + segment
        else:
            if current:
                parts.append(current)
            current = segment
    if current:
        parts.append(current)
    return parts
# Step 6: Compare two preparation approaches
fixed = fixed_chunks(clean, size=500)
structured = paragraph_chunks(clean, size=500)

print("\n--- Chunking Comparison ---")
print("Fixed chunks:", len(fixed))
print("Paragraph-aware chunks:", len(structured))

print("\nFixed sample:")
print(repr(fixed[0][:180]))

print("\nParagraph-aware sample:")
print(repr(structured[0][:180]))

print("\nPATHWAY A1 completed successfully.")

Source: Post Module Assessment_SAIA3453_final.pdf
Pages/segments: 6
Extracted characters: 12348
Cleaned characters: 10870
Fixed chunks: 22
Paragraph-aware chunks: 29
PATHWAY A1 completed successfully.


In [18]:
# Cell 5A2
# PATHWAY A2 — REQUIRED after A1
# Choose and export traceable text chunks
import json
# Step 1: Select the preparation approach
# Change to "fixed" if your Part A1 evidence supports it.
CHOSEN_METHOD = "paragraph-aware"
if CHOSEN_METHOD == "paragraph-aware":
    chosen = structured
    approach = "paragraph-aware-char-500"
elif CHOSEN_METHOD == "fixed":
    chosen = fixed
    approach = "fixed-char-500"
else:
    raise ValueError("Choose 'paragraph-aware' or 'fixed'")
# Step 2: Create traceable records
records = [
    {
        "record_id": f"{f.stem}-{i:04d}",
        "text": chunk,
        "source": str(f.relative_to(RAW_DIR)),
        "source_sha256": source_hash(f),
        "processing": approach,
        "page": None
    }
    for i, chunk in enumerate(chosen, 1)
]
# Page is None because the basic whole-document
# splitter does not preserve page boundaries.
# Step 3: Export prepared records
prepared_path = PROCESSED_DIR / "S04" / "prepared_text.jsonl"
prepared_path.parent.mkdir(parents=True, exist_ok=True)
with prepared_path.open("w", encoding="utf-8") as output:
    for row in records:
        output.write(
            json.dumps(row, ensure_ascii=False) + "\n"
        )
# Step 4: Basic validation
assert prepared_path.is_file(), "Prepared output not found"
assert prepared_path.stat().st_size > 0, "Prepared output is empty"
assert records, "No text chunks were produced"
assert all(row["text"].strip() for row in records), \
    "One or more chunks are empty"
with prepared_path.open("r", encoding="utf-8") as file:
    exported_records = [
        json.loads(line)
        for line in file
        if line.strip()
    ]
assert len(exported_records) == len(records), \
    "Exported record count does not match"

# Step 5: Display evidence
print("PATHWAY A2 completed successfully.")
print("Selected approach:", approach)
print("Total chunks exported:", len(records))
print("Prepared file:", prepared_path)
print("File size:", prepared_path.stat().st_size, "bytes")
print("Validation: PASS")
print("First record:", records[0])

PATHWAY A2 completed successfully.
Selected approach: paragraph-aware-char-500
Total chunks exported: 29
Prepared file: C:\vscode\oya\ai-engineering-utm-academicmate\data\processed\S04\prepared_text.jsonl
File size: 19110 bytes
Validation: PASS


In [19]:
assert prepared_path.exists(), 'Prepared output missing.'
assert prepared_path.stat().st_size > 0, 'Prepared output is empty.'
print('Output:',prepared_path, 'Size:',prepared_path.stat().st_size)
print('Validation: file exists and is non-empty; verify content manually too.')

Output: C:\vscode\oya\ai-engineering-utm-academicmate\data\processed\S04\prepared_text.jsonl Size: 19110
Validation: file exists and is non-empty; verify content manually too.


In [20]:
# Register Pathway A — PDF/TXT
register_lineage(
    source=f,
    output=prepared_path,
    processing=approach
)

Registered: Post Module Assessment_SAIA3453_final.pdf -> prepared_text.jsonl


### S05 · SAIA 2163 — Assignment 2_ Text Vectorization & Embedding Methods.pdf

Cells 5A1 and 5A2, pages 16–20. Compare fixed and paragraph-aware chunks. Then check and register this PDF on pages 35–36.

In [21]:
# Cell 5A1
# PATHWAY A1 — PDF/TXT
# CONDITIONAL REQUIRED: Run only for document-based projects
# Install PDF processing library
!pip -q install pypdf

from pypdf import PdfReader
import re

# Step 1: Select the exact source file
# Example: Synthetic practice PDF
# Replace with your actual project file where applicable.
f = select_source(
    relative_path=(
        "SAIA 2163 — Assignment 2_ Text Vectorization & Embedding Methods.pdf"
    )
)
# Step 2: Extract text according to file type
if f.suffix.lower() == ".pdf":
    reader = PdfReader(str(f))
    pages = [
        (i + 1, page.extract_text() or "")
        for i, page in enumerate(reader.pages)
    ]
elif f.suffix.lower() == ".txt":
    pages = [(1, f.read_text(encoding="utf-8"))]
else:
    raise ValueError("Pathway A needs a PDF or TXT source")
# Step 3: Basic text cleaning
text = "\n".join(content for _, content in pages)
clean = re.sub(r"[ \t]+", " ", text).strip()

print("Source:", f.name)
print("Pages/segments:", len(pages))
print("Extracted characters:", len(text))
print("Cleaned characters:", len(clean))
print("Content sample:", repr(clean[:300]))

assert clean, "No usable text: check extraction or OCR needs."
# Step 4: Fixed-size chunking
def fixed_chunks(s, size=500):
    if size <= 0:
        raise ValueError("Chunk size must be positive")
    return [
        s[i:i + size]
        for i in range(0, len(s), size)
    ]
# Step 5: Paragraph-aware chunking
def paragraph_chunks(s, size=500):
    if size <= 0:
        raise ValueError("Chunk size must be positive")
    parts = []
    current = ""
    for segment in re.split(r"\n\s*\n", s):
        segment = segment.strip()
        if not segment:
            continue
        if len(segment) > size:
            if current:
                parts.append(current)
                current = ""
            parts.extend(fixed_chunks(segment, size))

        elif len(current) + len(segment) + (2 if current else 0) <= size:
            current += ("\n\n" if current else "") + segment
        else:
            if current:
                parts.append(current)
            current = segment
    if current:
        parts.append(current)
    return parts
# Step 6: Compare two preparation approaches
fixed = fixed_chunks(clean, size=500)
structured = paragraph_chunks(clean, size=500)

print("\n--- Chunking Comparison ---")
print("Fixed chunks:", len(fixed))
print("Paragraph-aware chunks:", len(structured))

print("\nFixed sample:")
print(repr(fixed[0][:180]))

print("\nParagraph-aware sample:")
print(repr(structured[0][:180]))

print("\nPATHWAY A1 completed successfully.")

Source: SAIA 2163 — Assignment 2_ Text Vectorization & Embedding Methods.pdf
Pages/segments: 15
Extracted characters: 20773
Cleaned characters: 20752
Fixed chunks: 42
Paragraph-aware chunks: 42
PATHWAY A1 completed successfully.


In [22]:
# Cell 5A2
# PATHWAY A2 — REQUIRED after A1
# Choose and export traceable text chunks
import json
# Step 1: Select the preparation approach
# Change to "fixed" if your Part A1 evidence supports it.
CHOSEN_METHOD = "paragraph-aware"
if CHOSEN_METHOD == "paragraph-aware":
    chosen = structured
    approach = "paragraph-aware-char-500"
elif CHOSEN_METHOD == "fixed":
    chosen = fixed
    approach = "fixed-char-500"
else:
    raise ValueError("Choose 'paragraph-aware' or 'fixed'")
# Step 2: Create traceable records
records = [
    {
        "record_id": f"{f.stem}-{i:04d}",
        "text": chunk,
        "source": str(f.relative_to(RAW_DIR)),
        "source_sha256": source_hash(f),
        "processing": approach,
        "page": None
    }
    for i, chunk in enumerate(chosen, 1)
]
# Page is None because the basic whole-document
# splitter does not preserve page boundaries.
# Step 3: Export prepared records
prepared_path = PROCESSED_DIR / "S05" / "prepared_text.jsonl"
prepared_path.parent.mkdir(parents=True, exist_ok=True)
with prepared_path.open("w", encoding="utf-8") as output:
    for row in records:
        output.write(
            json.dumps(row, ensure_ascii=False) + "\n"
        )
# Step 4: Basic validation
assert prepared_path.is_file(), "Prepared output not found"
assert prepared_path.stat().st_size > 0, "Prepared output is empty"
assert records, "No text chunks were produced"
assert all(row["text"].strip() for row in records), \
    "One or more chunks are empty"
with prepared_path.open("r", encoding="utf-8") as file:
    exported_records = [
        json.loads(line)
        for line in file
        if line.strip()
    ]
assert len(exported_records) == len(records), \
    "Exported record count does not match"

# Step 5: Display evidence
print("PATHWAY A2 completed successfully.")
print("Selected approach:", approach)
print("Total chunks exported:", len(records))
print("Prepared file:", prepared_path)
print("File size:", prepared_path.stat().st_size, "bytes")
print("Validation: PASS")
print("First record:", records[0])

PATHWAY A2 completed successfully.
Selected approach: paragraph-aware-char-500
Total chunks exported: 42
Prepared file: C:\vscode\oya\ai-engineering-utm-academicmate\data\processed\S05\prepared_text.jsonl
File size: 35203 bytes
Validation: PASS


In [23]:
assert prepared_path.exists(), 'Prepared output missing.'
assert prepared_path.stat().st_size > 0, 'Prepared output is empty.'
print('Output:',prepared_path, 'Size:',prepared_path.stat().st_size)
print('Validation: file exists and is non-empty; verify content manually too.')

Output: C:\vscode\oya\ai-engineering-utm-academicmate\data\processed\S05\prepared_text.jsonl Size: 35203
Validation: file exists and is non-empty; verify content manually too.


In [24]:
# Register Pathway A — PDF/TXT
register_lineage(
    source=f,
    output=prepared_path,
    processing=approach
)

Registered: SAIA 2163 — Assignment 2_ Text Vectorization & Embedding Methods.pdf -> prepared_text.jsonl


### S06 and S07 · Compare screenshot sizes

Cell 5C, pages 24–26. Run the example size first. The next cell writes the larger copies to the same paths before they are registered. Small text is easier to read on the larger images.

In [25]:
# Cell 5C
# PATHWAY C — IMAGE DATA
# CONDITIONAL REQUIRED: Run only for computer-vision projects
from PIL import Image, UnidentifiedImageError
import pandas as pd
# Step 1: Select the source image folder
# Replace with your actual project image folder where applicable.
source_dir = RAW_DIR / "."
assert source_dir.is_dir(), (
    f"Choose an existing image folder: {source_dir}"
)
# Step 2: Collect supported image files
image_files = sorted(
    p for p in source_dir.iterdir()
    if p.is_file() and p.suffix.lower() in {
        ".png", ".jpg", ".jpeg", ".webp"
    }
)
assert image_files, "No images found in the selected folder"
print("--- Source Image Folder ---")
print("Folder:", source_dir)
print("Image count:", len(image_files))
print("Example files:", [p.name for p in image_files[:5]])
# Step 3: Configure preparation settings
# Change these only if justified by your Part B decision.
TARGET_SIZE = (224, 224)
CONVERT_TO_RGB = True
# Step 4: Prepare output folder
image_dir = PROCESSED_DIR / "images"
image_dir.mkdir(parents=True, exist_ok=True)
rows = []
failed_files = []
# Step 5: Process images
for f in image_files:
    try:
        with Image.open(f) as im:
            im.load()
            original_size = im.size
            original_mode = im.mode
            transformed = im
            if CONVERT_TO_RGB:
                transformed = transformed.convert("RGB")
            transformed = transformed.resize(TARGET_SIZE)
            target = image_dir / f"{f.stem}_{source_hash(f)[:12]}.png"
            transformed.save(target)
        rows.append({
            "source": str(f.relative_to(RAW_DIR)),
            "source_sha256": source_hash(f),
            "original_width": original_size[0],
            "original_height": original_size[1],
            "original_mode": original_mode,
            "output": str(target.relative_to(BASE_DIR)),
            "output_width": TARGET_SIZE[0],
            "output_height": TARGET_SIZE[1],
            "processing": (
                f"convert_to_rgb={CONVERT_TO_RGB}; "
                f"resize={TARGET_SIZE}"
            )
        })
    except (UnidentifiedImageError, OSError) as err:
        failed_files.append({
            "file": f.name,
            "error": str(err)
        })
        print("Failed:", f.name, "|", str(err))
# Step 6: Validate processing results
assert rows, "No readable images were prepared"
prepared_path = PROCESSED_DIR / "image_manifest.csv"
pd.DataFrame(rows).to_csv(prepared_path, index=False)
assert prepared_path.is_file(), "Image manifest was not created"
assert prepared_path.stat().st_size > 0, "Image manifest is empty"
# Step 7: Show evidence
print("\n--- Preparation Summary ---")
print("Images prepared:", len(rows))
print("Images failed:", len(failed_files))
print("Prepared image folder:", image_dir)
print("Manifest:", prepared_path)
manifest_df = pd.read_csv(prepared_path)
print("\nManifest preview:")
print(manifest_df.head().to_string(index=False))
# Step 8: Important note for labels
print("\nNote:")
print(
    "If your project uses labels or annotations "
    "(for example labels.csv, bounding boxes, or masks), "
    "they remain raw unless you explicitly process them."
)
print(
    "Do not infer labels from filenames unless this is "
    "justified by your dataset design."
)
# Preserve the image folder reference for Step B4
f = source_dir
print("\nPATHWAY C completed successfully.")

--- Source Image Folder ---
Folder: C:\vscode\oya\ai-engineering-utm-academicmate\data\raw
Image count: 2
Example files: ['Screenshot 2026-11-02 093501.png', 'Screenshot 2026-11-02 094157.png']

--- Preparation Summary ---
Images prepared: 2
Images failed: 0
Prepared image folder: C:\vscode\oya\ai-engineering-utm-academicmate\data\processed\images
Manifest: C:\vscode\oya\ai-engineering-utm-academicmate\data\processed\image_manifest.csv

Manifest preview:
                          source                                                    source_sha256  original_width  original_height original_mode                                                              output  output_width  output_height                             processing
Screenshot 2026-11-02 093501.png 2f792dced400a5e31939030517c406080b6020afe1a965299f0fb79776299062            1413              767          RGBA data\processed\images\Screenshot 2026-11-02 093501_2f792dced400.png           224            224 convert_to_rgb=Tru

**Selected size.** Run Cell 5C again at 1413 × 837. Both time labels are readable at this size. Images with different shapes stretch slightly.

In [26]:
# Cell 5C
# PATHWAY C — IMAGE DATA
# CONDITIONAL REQUIRED: Run only for computer-vision projects
from PIL import Image, UnidentifiedImageError
import pandas as pd
# Step 1: Select the source image folder
# Replace with your actual project image folder where applicable.
source_dir = RAW_DIR / "."
assert source_dir.is_dir(), (
    f"Choose an existing image folder: {source_dir}"
)
# Step 2: Collect supported image files
image_files = sorted(
    p for p in source_dir.iterdir()
    if p.is_file() and p.suffix.lower() in {
        ".png", ".jpg", ".jpeg", ".webp"
    }
)
assert image_files, "No images found in the selected folder"
print("--- Source Image Folder ---")
print("Folder:", source_dir)
print("Image count:", len(image_files))
print("Example files:", [p.name for p in image_files[:5]])
# Step 3: Configure preparation settings
# Change these only if justified by your Part B decision.
TARGET_SIZE = (1413, 837)
CONVERT_TO_RGB = True
# Step 4: Prepare output folder
image_dir = PROCESSED_DIR / "images"
image_dir.mkdir(parents=True, exist_ok=True)
rows = []
failed_files = []
# Step 5: Process images
for f in image_files:
    try:
        with Image.open(f) as im:
            im.load()
            original_size = im.size
            original_mode = im.mode
            transformed = im
            if CONVERT_TO_RGB:
                transformed = transformed.convert("RGB")
            transformed = transformed.resize(TARGET_SIZE)
            target = image_dir / f"{f.stem}_{source_hash(f)[:12]}.png"
            transformed.save(target)
        rows.append({
            "source": str(f.relative_to(RAW_DIR)),
            "source_sha256": source_hash(f),
            "original_width": original_size[0],
            "original_height": original_size[1],
            "original_mode": original_mode,
            "output": str(target.relative_to(BASE_DIR)),
            "output_width": TARGET_SIZE[0],
            "output_height": TARGET_SIZE[1],
            "processing": (
                f"convert_to_rgb={CONVERT_TO_RGB}; "
                f"resize={TARGET_SIZE}"
            )
        })
    except (UnidentifiedImageError, OSError) as err:
        failed_files.append({
            "file": f.name,
            "error": str(err)
        })
        print("Failed:", f.name, "|", str(err))
# Step 6: Validate processing results
assert rows, "No readable images were prepared"
prepared_path = PROCESSED_DIR / "image_manifest.csv"
pd.DataFrame(rows).to_csv(prepared_path, index=False)
assert prepared_path.is_file(), "Image manifest was not created"
assert prepared_path.stat().st_size > 0, "Image manifest is empty"
# Step 7: Show evidence
print("\n--- Preparation Summary ---")
print("Images prepared:", len(rows))
print("Images failed:", len(failed_files))
print("Prepared image folder:", image_dir)
print("Manifest:", prepared_path)
manifest_df = pd.read_csv(prepared_path)
print("\nManifest preview:")
print(manifest_df.head().to_string(index=False))
# Step 8: Important note for labels
print("\nNote:")
print(
    "If your project uses labels or annotations "
    "(for example labels.csv, bounding boxes, or masks), "
    "they remain raw unless you explicitly process them."
)
print(
    "Do not infer labels from filenames unless this is "
    "justified by your dataset design."
)
# Preserve the image folder reference for Step B4
f = source_dir
print("\nPATHWAY C completed successfully.")

--- Source Image Folder ---
Folder: C:\vscode\oya\ai-engineering-utm-academicmate\data\raw
Image count: 2
Example files: ['Screenshot 2026-11-02 093501.png', 'Screenshot 2026-11-02 094157.png']

--- Preparation Summary ---
Images prepared: 2
Images failed: 0
Prepared image folder: C:\vscode\oya\ai-engineering-utm-academicmate\data\processed\images
Manifest: C:\vscode\oya\ai-engineering-utm-academicmate\data\processed\image_manifest.csv

Manifest preview:
                          source                                                    source_sha256  original_width  original_height original_mode                                                              output  output_width  output_height                              processing
Screenshot 2026-11-02 093501.png 2f792dced400a5e31939030517c406080b6020afe1a965299f0fb79776299062            1413              767          RGBA data\processed\images\Screenshot 2026-11-02 093501_2f792dced400.png          1413            837 convert_to_rgb=Tr

In [27]:
assert prepared_path.exists(), 'Prepared output missing.'
assert prepared_path.stat().st_size > 0, 'Prepared output is empty.'
print('Output:',prepared_path, 'Size:',prepared_path.stat().st_size)
print('Validation: file exists and is non-empty; verify content manually too.')

Output: C:\vscode\oya\ai-engineering-utm-academicmate\data\processed\image_manifest.csv Size: 576
Validation: file exists and is non-empty; verify content manually too.


In [28]:
# Register Pathway C — Images
for row in rows:
    register_lineage(
        source=RAW_DIR / row["source"],
        output=BASE_DIR / row["output"],
        processing=row["processing"]
    )

Registered: Screenshot 2026-11-02 093501.png -> Screenshot 2026-11-02 093501_2f792dced400.png
Registered: Screenshot 2026-11-02 094157.png -> Screenshot 2026-11-02 094157_99576e558a01.png


### Check the common manifest

Cells 6A and 6B, pages 38–40. Expect seven links, one per input.

In [29]:
# CELL 6A — GENERATE COMMON LINEAGE MANIFEST
# REQUIRED: Run after all selected pathways are registered
assert lineage_entries, (
    "No lineage entries registered. "
    "Register at least one selected pathway first."
)
lineage_records = []

for i, entry in enumerate(lineage_entries.values(), 1):

    source = entry["source"]
    output = entry["output"]
    processing = entry["processing"]

    # Validate registered files again
    assert source.is_file(), f"Missing source: {source}"
    assert output.is_file(), f"Missing output: {output}"
    assert output.stat().st_size > 0, "Empty output"

    lineage_records.append({
        "source_id": f"S{i:02d}",
        "source_filename": str(
            source.relative_to(RAW_DIR.resolve())
        ),
        "source_sha256": source_hash(source),
        "prepared_output": str(
            output.relative_to(BASE_DIR.resolve())
        ),
        "prepared_sha256": source_hash(output),
        "processing_note": processing,
        "source_version": "Not verified",
        "access_rights": "Not verified",
        "status": "Prepared; quality review pending"
    })

# Reuse the common manifest writer from Cell 4
manifest = save_manifest(lineage_records)

print("\nCOMMON LINEAGE MANIFEST GENERATED")
print("Total lineage entries:", len(manifest["records"]))
print(
    "Saved to:",
    BASE_DIR / "data" / "week3_manifest.json"
)

Saved lineage: C:\vscode\oya\ai-engineering-utm-academicmate\data\week3_manifest.json

COMMON LINEAGE MANIFEST GENERATED
Total lineage entries: 7
Saved to: C:\vscode\oya\ai-engineering-utm-academicmate\data\week3_manifest.json


In [30]:
# CELL 6B — VERIFY COMMON LINEAGE MANIFEST
# REQUIRED: Run after Cell 6A
import json
manifest_path = BASE_DIR / "data" / "week3_manifest.json"
assert manifest_path.is_file(), (
    "Common lineage manifest not found."
)
with manifest_path.open("r", encoding="utf-8") as file:
    saved_manifest = json.load(file)
saved_records = saved_manifest.get("records", [])
assert saved_records, (
    "No lineage records found in saved manifest."
)
assert len(saved_records) == len(lineage_entries), (
    "Saved lineage count does not match registration."
)
for record in saved_records:
    source = RAW_DIR / record["source_filename"]
    output = BASE_DIR / record["prepared_output"]
    assert source.is_file(), f"Missing source: {source}"
    assert output.is_file(), f"Missing output: {output}"
    assert source_hash(source) == record["source_sha256"], (
        f"Source checksum mismatch: {source}"
    )
    assert source_hash(output) == record["prepared_sha256"], (
        f"Prepared output checksum mismatch: {output}"
    )
print("COMMON LINEAGE VALIDATION: PASS")
print("Verified records:", len(saved_records))
print("Manifest:", manifest_path)

COMMON LINEAGE VALIDATION: PASS
Verified records: 7
Manifest: C:\vscode\oya\ai-engineering-utm-academicmate\data\week3_manifest.json


## Part C · Build and repeat the pipeline

Cell 7, pages 42–46. It records file hashes and keeps each run for the final check.

In [31]:
# Cell 7 — REQUIRED: Common V1 Pipeline Utilities
# Run ONCE after completing Part B

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json

# Step 1: Define standard project paths
BASE_DIR = Path("/content/week3_project")
RAW_DIR = BASE_DIR / "data" / "raw"
PIPE_DIR = BASE_DIR / "data" / "pipeline_v1"

PIPE_DIR.mkdir(parents=True, exist_ok=True)

# Store registered pipeline runs for repeatability verification
PIPELINE_RUNS = {}


# Step 2: Calculate SHA-256 checksum
def sha256_file(path):
    return hashlib.sha256(
        Path(path).read_bytes()
    ).hexdigest()


# Step 3: Resolve an explicitly selected source
def resolve_source(relative_path, folder=False):
    source = RAW_DIR / relative_path

    if folder:
        if not source.is_dir():
            raise FileNotFoundError(
                f"Source folder not found: {source}"
            )
    else:
        if not source.is_file():
            raise FileNotFoundError(
                f"Source file not found: {source}"
            )

    return source


# Step 4: Execute the selected V1 data pipeline
def execute_pipeline(
    kind,
    source,
    inspect_fn,
    prepare_fn,
    configuration
):
    source = Path(source)

    # INGEST: Identify relevant source files
    if kind == "images":
        inputs = sorted(
            p for p in source.iterdir()
            if p.is_file()
            and p.suffix.lower() in {
                ".png", ".jpg", ".jpeg", ".webp"
            }
        )
    else:
        inputs = [source]

    if not inputs or any(not p.is_file() for p in inputs):
        raise ValueError(
            "No valid source files to ingest."
        )

    # INSPECT: Apply data-type-specific inspection
    inspection = inspect_fn(inputs, configuration)

    # Define output directory
    output_dir = PIPE_DIR / kind
    output_dir.mkdir(parents=True, exist_ok=True)

    # PREPARE / STORE: Apply selected transformation
    outputs, mappings, method = prepare_fn(
        inputs,
        output_dir,
        configuration
    )
    outputs = [Path(p) for p in outputs]

    # VALIDATE: Check prepared outputs
    if not outputs:
        raise ValueError(
            "No prepared outputs were produced."
        )

    for output in outputs:
        if not output.is_file():
            raise FileNotFoundError(
                f"Prepared output missing: {output}"
            )

        if output.stat().st_size == 0:
            raise ValueError(
                f"Prepared output is empty: {output}"
            )

    # Record basic validation evidence
    validation = {
        "output_count": len(outputs),
        "output_sha256": [
            sha256_file(p) for p in outputs
        ],
        "output_bytes": [
            p.stat().st_size for p in outputs
        ]
    }

    # RECORD LINEAGE: Create pipeline execution record
    record = {
        "pipeline_version": "week3-v1",
        "generated_at_utc": datetime.now(
            timezone.utc
        ).isoformat(),
        "data_type": kind,
        "configuration": configuration,
        "processing_method": method,
        "sources": [
            {
                "path": str(p),
                "sha256": sha256_file(p)
            }
            for p in inputs
        ],
        "outputs": [
            str(p) for p in outputs
        ],
        "source_output_mapping": mappings,
        "inspection": inspection,
        "validation": validation,
        "source_version": "Not verified",
        "access_rights": (
            "Not verified — check before sharing"
        )
    }

    # Save pipeline manifest
    manifest_path = output_dir / "pipeline_manifest.json"

    manifest_path.write_text(
        json.dumps(
            record,
            indent=2,
            ensure_ascii=False
        ),
        encoding="utf-8"
    )

    print(
        f"SUCCESS: {kind} | "
        f"Inputs: {len(inputs)} | "
        f"Outputs: {len(outputs)}"
    )
    print("Manifest:", manifest_path)

    return record


# Step 5: Register and run a selected pathway
def register_and_run(
    kind,
    source,
    inspect_fn,
    prepare_fn,
    configuration
):
    # Keep the runner for repeatability verification
    def runner():
        return execute_pipeline(
            kind,
            source,
            inspect_fn,
            prepare_fn,
            configuration
        )

    first = runner()

    PIPELINE_RUNS[kind] = {
        "first": first,
        "runner": runner
    }
    return first
print("Cell 7 ready.")
print("Run ONLY the selected Cell 8 pathway(s).")
print("Common pipeline utilities are available.")

Cell 7 ready.
Run ONLY the selected Cell 8 pathway(s).
Common pipeline utilities are available.


### S01 · Activity for Module 2.pdf

Cell 8A, pages 46–49. Each PDF gets its own results and run record.

In [32]:
# Cell 8A — CONDITIONAL REQUIRED: PDF/TXT Projects
# Run only if your project uses document data.
# Requires Cell 7 (Common V1 Pipeline Utilities).

!pip -q install pypdf

import re
import json
from pypdf import PdfReader

# Step 1: Select the source and preparation configuration
# Replace with your actual source and Part B decisions.

SOURCE_A = resolve_source(
    "Activity for Module 2.pdf"
)

CONFIG_A = {
    "chunk_size": 500,
    "method": "paragraph-aware"
}

# Step 2: Inspect document content
def inspect_documents(inputs, cfg):
    src = inputs[0]
    if src.suffix.lower() == ".pdf":
        reader = PdfReader(str(src))
        pages = [
            page.extract_text() or ""
            for page in reader.pages
        ]
        text = "\n".join(pages)
        page_count = len(pages)

    elif src.suffix.lower() == ".txt":
        text = src.read_text(encoding="utf-8")
        page_count = None

    else:
        raise ValueError("Expected a PDF or TXT source")

    if not text.strip():
        raise ValueError(
            "No extractable text found. "
            "Scanned PDFs may require OCR."
        )

    return {
        "source": src.name,
        "characters": len(text),
        "pages": page_count
    }


# Step 3: Define document preparation
def prepare_documents(inputs, out, cfg):
    src = inputs[0]

    # Extract text
    if src.suffix.lower() == ".pdf":
        reader = PdfReader(str(src))
        text = "\n".join(
            page.extract_text() or ""
            for page in reader.pages
        )

    elif src.suffix.lower() == ".txt":
        text = src.read_text(encoding="utf-8")

    else:
        raise ValueError("Expected a PDF or TXT source")

    # Apply the same cleaning used in Part B
    text = re.sub(r"[ \t]+", " ", text).strip()

    size = int(cfg["chunk_size"])
    method = cfg["method"]

    if size <= 0:
        raise ValueError("Chunk size must be positive")

    # Fixed-size chunking
    if method == "fixed":
        chunks = [
            text[i:i + size]
            for i in range(0, len(text), size)
        ]

    # Paragraph-aware chunking
    elif method == "paragraph-aware":
        chunks = []
        current = ""

        for section in re.split(r"\n\s*\n", text):
            section = section.strip()

            if not section:
                continue

            if len(section) > size:
                if current:
                    chunks.append(current)
                    current = ""

                chunks.extend(
                    section[i:i + size]
                    for i in range(0, len(section), size)
                )

            elif len(current) + len(section) + (2 if current else 0) <= size:
                current += (
                    "\n\n" if current else ""
                ) + section

            else:
                if current:
                    chunks.append(current)

                current = section

        if current:
            chunks.append(current)

    else:
        raise ValueError(
            "Choose 'fixed' or 'paragraph-aware'"
        )

    if not chunks:
        raise ValueError("No text chunks produced")

    # Step 4: Store prepared document records
    target = out / "prepared_text.jsonl"

    with target.open("w", encoding="utf-8") as handle:
        for i, chunk in enumerate(chunks, 1):
            record = {
                "id": i,
                "source": str(src),
                "text": chunk,
                "page": None
            }
            handle.write(
                json.dumps(record, ensure_ascii=False) + "\n"
            )

    # Step 5: Return source-to-output mapping
    mapping = [
        {
            "source": str(src),
            "output": str(target)
        }
    ]

    processing_method = (
        f"{method} text chunks of {size} characters"
    )

    return [target], mapping, processing_method

# Step 6: Execute and register the pipeline
run_A = register_and_run(
    "documents/S01",
    SOURCE_A,
    inspect_documents,
    prepare_documents,
    CONFIG_A
)
print("Cell 8A completed successfully.")

SUCCESS: documents/S01 | Inputs: 1 | Outputs: 1
Manifest: C:\vscode\oya\ai-engineering-utm-academicmate\data\pipeline_v1\documents\S01\pipeline_manifest.json
Cell 8A completed successfully.


### S02 · Assignment 1[Individual].pdf

Cell 8A, pages 46–49. Each PDF gets its own results and run record.

In [33]:
# Cell 8A — CONDITIONAL REQUIRED: PDF/TXT Projects
# Run only if your project uses document data.
# Requires Cell 7 (Common V1 Pipeline Utilities).

!pip -q install pypdf

import re
import json
from pypdf import PdfReader

# Step 1: Select the source and preparation configuration
# Replace with your actual source and Part B decisions.

SOURCE_A = resolve_source(
    "Assignment 1[Individual].pdf"
)

CONFIG_A = {
    "chunk_size": 500,
    "method": "paragraph-aware"
}

# Step 2: Inspect document content
def inspect_documents(inputs, cfg):
    src = inputs[0]
    if src.suffix.lower() == ".pdf":
        reader = PdfReader(str(src))
        pages = [
            page.extract_text() or ""
            for page in reader.pages
        ]
        text = "\n".join(pages)
        page_count = len(pages)

    elif src.suffix.lower() == ".txt":
        text = src.read_text(encoding="utf-8")
        page_count = None

    else:
        raise ValueError("Expected a PDF or TXT source")

    if not text.strip():
        raise ValueError(
            "No extractable text found. "
            "Scanned PDFs may require OCR."
        )

    return {
        "source": src.name,
        "characters": len(text),
        "pages": page_count
    }


# Step 3: Define document preparation
def prepare_documents(inputs, out, cfg):
    src = inputs[0]

    # Extract text
    if src.suffix.lower() == ".pdf":
        reader = PdfReader(str(src))
        text = "\n".join(
            page.extract_text() or ""
            for page in reader.pages
        )

    elif src.suffix.lower() == ".txt":
        text = src.read_text(encoding="utf-8")

    else:
        raise ValueError("Expected a PDF or TXT source")

    # Apply the same cleaning used in Part B
    text = re.sub(r"[ \t]+", " ", text).strip()

    size = int(cfg["chunk_size"])
    method = cfg["method"]

    if size <= 0:
        raise ValueError("Chunk size must be positive")

    # Fixed-size chunking
    if method == "fixed":
        chunks = [
            text[i:i + size]
            for i in range(0, len(text), size)
        ]

    # Paragraph-aware chunking
    elif method == "paragraph-aware":
        chunks = []
        current = ""

        for section in re.split(r"\n\s*\n", text):
            section = section.strip()

            if not section:
                continue

            if len(section) > size:
                if current:
                    chunks.append(current)
                    current = ""

                chunks.extend(
                    section[i:i + size]
                    for i in range(0, len(section), size)
                )

            elif len(current) + len(section) + (2 if current else 0) <= size:
                current += (
                    "\n\n" if current else ""
                ) + section

            else:
                if current:
                    chunks.append(current)

                current = section

        if current:
            chunks.append(current)

    else:
        raise ValueError(
            "Choose 'fixed' or 'paragraph-aware'"
        )

    if not chunks:
        raise ValueError("No text chunks produced")

    # Step 4: Store prepared document records
    target = out / "prepared_text.jsonl"

    with target.open("w", encoding="utf-8") as handle:
        for i, chunk in enumerate(chunks, 1):
            record = {
                "id": i,
                "source": str(src),
                "text": chunk,
                "page": None
            }
            handle.write(
                json.dumps(record, ensure_ascii=False) + "\n"
            )

    # Step 5: Return source-to-output mapping
    mapping = [
        {
            "source": str(src),
            "output": str(target)
        }
    ]

    processing_method = (
        f"{method} text chunks of {size} characters"
    )

    return [target], mapping, processing_method

# Step 6: Execute and register the pipeline
run_A = register_and_run(
    "documents",
    SOURCE_A,
    inspect_documents,
    prepare_documents,
    CONFIG_A
)
print("Cell 8A completed successfully.")

SUCCESS: documents | Inputs: 1 | Outputs: 1
Manifest: C:\vscode\oya\ai-engineering-utm-academicmate\data\pipeline_v1\documents\pipeline_manifest.json
Cell 8A completed successfully.


### S03 · NLP Final Project Question.pdf

Cell 8A, pages 46–49. Each PDF gets its own results and run record.

In [34]:
# Cell 8A — CONDITIONAL REQUIRED: PDF/TXT Projects
# Run only if your project uses document data.
# Requires Cell 7 (Common V1 Pipeline Utilities).

!pip -q install pypdf

import re
import json
from pypdf import PdfReader

# Step 1: Select the source and preparation configuration
# Replace with your actual source and Part B decisions.

SOURCE_A = resolve_source(
    "NLP Final Project Question.pdf"
)

CONFIG_A = {
    "chunk_size": 500,
    "method": "paragraph-aware"
}

# Step 2: Inspect document content
def inspect_documents(inputs, cfg):
    src = inputs[0]
    if src.suffix.lower() == ".pdf":
        reader = PdfReader(str(src))
        pages = [
            page.extract_text() or ""
            for page in reader.pages
        ]
        text = "\n".join(pages)
        page_count = len(pages)

    elif src.suffix.lower() == ".txt":
        text = src.read_text(encoding="utf-8")
        page_count = None

    else:
        raise ValueError("Expected a PDF or TXT source")

    if not text.strip():
        raise ValueError(
            "No extractable text found. "
            "Scanned PDFs may require OCR."
        )

    return {
        "source": src.name,
        "characters": len(text),
        "pages": page_count
    }


# Step 3: Define document preparation
def prepare_documents(inputs, out, cfg):
    src = inputs[0]

    # Extract text
    if src.suffix.lower() == ".pdf":
        reader = PdfReader(str(src))
        text = "\n".join(
            page.extract_text() or ""
            for page in reader.pages
        )

    elif src.suffix.lower() == ".txt":
        text = src.read_text(encoding="utf-8")

    else:
        raise ValueError("Expected a PDF or TXT source")

    # Apply the same cleaning used in Part B
    text = re.sub(r"[ \t]+", " ", text).strip()

    size = int(cfg["chunk_size"])
    method = cfg["method"]

    if size <= 0:
        raise ValueError("Chunk size must be positive")

    # Fixed-size chunking
    if method == "fixed":
        chunks = [
            text[i:i + size]
            for i in range(0, len(text), size)
        ]

    # Paragraph-aware chunking
    elif method == "paragraph-aware":
        chunks = []
        current = ""

        for section in re.split(r"\n\s*\n", text):
            section = section.strip()

            if not section:
                continue

            if len(section) > size:
                if current:
                    chunks.append(current)
                    current = ""

                chunks.extend(
                    section[i:i + size]
                    for i in range(0, len(section), size)
                )

            elif len(current) + len(section) + (2 if current else 0) <= size:
                current += (
                    "\n\n" if current else ""
                ) + section

            else:
                if current:
                    chunks.append(current)

                current = section

        if current:
            chunks.append(current)

    else:
        raise ValueError(
            "Choose 'fixed' or 'paragraph-aware'"
        )

    if not chunks:
        raise ValueError("No text chunks produced")

    # Step 4: Store prepared document records
    target = out / "prepared_text.jsonl"

    with target.open("w", encoding="utf-8") as handle:
        for i, chunk in enumerate(chunks, 1):
            record = {
                "id": i,
                "source": str(src),
                "text": chunk,
                "page": None
            }
            handle.write(
                json.dumps(record, ensure_ascii=False) + "\n"
            )

    # Step 5: Return source-to-output mapping
    mapping = [
        {
            "source": str(src),
            "output": str(target)
        }
    ]

    processing_method = (
        f"{method} text chunks of {size} characters"
    )

    return [target], mapping, processing_method

# Step 6: Execute and register the pipeline
run_A = register_and_run(
    "documents/S03",
    SOURCE_A,
    inspect_documents,
    prepare_documents,
    CONFIG_A
)
print("Cell 8A completed successfully.")

SUCCESS: documents/S03 | Inputs: 1 | Outputs: 1
Manifest: C:\vscode\oya\ai-engineering-utm-academicmate\data\pipeline_v1\documents\S03\pipeline_manifest.json
Cell 8A completed successfully.


### S04 · Post Module Assessment_SAIA3453_final.pdf

Cell 8A, pages 46–49. Each PDF gets its own results and run record.

In [35]:
# Cell 8A — CONDITIONAL REQUIRED: PDF/TXT Projects
# Run only if your project uses document data.
# Requires Cell 7 (Common V1 Pipeline Utilities).

!pip -q install pypdf

import re
import json
from pypdf import PdfReader

# Step 1: Select the source and preparation configuration
# Replace with your actual source and Part B decisions.

SOURCE_A = resolve_source(
    "Post Module Assessment_SAIA3453_final.pdf"
)

CONFIG_A = {
    "chunk_size": 500,
    "method": "paragraph-aware"
}

# Step 2: Inspect document content
def inspect_documents(inputs, cfg):
    src = inputs[0]
    if src.suffix.lower() == ".pdf":
        reader = PdfReader(str(src))
        pages = [
            page.extract_text() or ""
            for page in reader.pages
        ]
        text = "\n".join(pages)
        page_count = len(pages)

    elif src.suffix.lower() == ".txt":
        text = src.read_text(encoding="utf-8")
        page_count = None

    else:
        raise ValueError("Expected a PDF or TXT source")

    if not text.strip():
        raise ValueError(
            "No extractable text found. "
            "Scanned PDFs may require OCR."
        )

    return {
        "source": src.name,
        "characters": len(text),
        "pages": page_count
    }


# Step 3: Define document preparation
def prepare_documents(inputs, out, cfg):
    src = inputs[0]

    # Extract text
    if src.suffix.lower() == ".pdf":
        reader = PdfReader(str(src))
        text = "\n".join(
            page.extract_text() or ""
            for page in reader.pages
        )

    elif src.suffix.lower() == ".txt":
        text = src.read_text(encoding="utf-8")

    else:
        raise ValueError("Expected a PDF or TXT source")

    # Apply the same cleaning used in Part B
    text = re.sub(r"[ \t]+", " ", text).strip()

    size = int(cfg["chunk_size"])
    method = cfg["method"]

    if size <= 0:
        raise ValueError("Chunk size must be positive")

    # Fixed-size chunking
    if method == "fixed":
        chunks = [
            text[i:i + size]
            for i in range(0, len(text), size)
        ]

    # Paragraph-aware chunking
    elif method == "paragraph-aware":
        chunks = []
        current = ""

        for section in re.split(r"\n\s*\n", text):
            section = section.strip()

            if not section:
                continue

            if len(section) > size:
                if current:
                    chunks.append(current)
                    current = ""

                chunks.extend(
                    section[i:i + size]
                    for i in range(0, len(section), size)
                )

            elif len(current) + len(section) + (2 if current else 0) <= size:
                current += (
                    "\n\n" if current else ""
                ) + section

            else:
                if current:
                    chunks.append(current)

                current = section

        if current:
            chunks.append(current)

    else:
        raise ValueError(
            "Choose 'fixed' or 'paragraph-aware'"
        )

    if not chunks:
        raise ValueError("No text chunks produced")

    # Step 4: Store prepared document records
    target = out / "prepared_text.jsonl"

    with target.open("w", encoding="utf-8") as handle:
        for i, chunk in enumerate(chunks, 1):
            record = {
                "id": i,
                "source": str(src),
                "text": chunk,
                "page": None
            }
            handle.write(
                json.dumps(record, ensure_ascii=False) + "\n"
            )

    # Step 5: Return source-to-output mapping
    mapping = [
        {
            "source": str(src),
            "output": str(target)
        }
    ]

    processing_method = (
        f"{method} text chunks of {size} characters"
    )

    return [target], mapping, processing_method

# Step 6: Execute and register the pipeline
run_A = register_and_run(
    "documents/S04",
    SOURCE_A,
    inspect_documents,
    prepare_documents,
    CONFIG_A
)
print("Cell 8A completed successfully.")

SUCCESS: documents/S04 | Inputs: 1 | Outputs: 1
Manifest: C:\vscode\oya\ai-engineering-utm-academicmate\data\pipeline_v1\documents\S04\pipeline_manifest.json
Cell 8A completed successfully.


### S05 · SAIA 2163 — Assignment 2_ Text Vectorization & Embedding Methods.pdf

Cell 8A, pages 46–49. Each PDF gets its own results and run record.

In [36]:
# Cell 8A — CONDITIONAL REQUIRED: PDF/TXT Projects
# Run only if your project uses document data.
# Requires Cell 7 (Common V1 Pipeline Utilities).

!pip -q install pypdf

import re
import json
from pypdf import PdfReader

# Step 1: Select the source and preparation configuration
# Replace with your actual source and Part B decisions.

SOURCE_A = resolve_source(
    "SAIA 2163 — Assignment 2_ Text Vectorization & Embedding Methods.pdf"
)

CONFIG_A = {
    "chunk_size": 500,
    "method": "paragraph-aware"
}

# Step 2: Inspect document content
def inspect_documents(inputs, cfg):
    src = inputs[0]
    if src.suffix.lower() == ".pdf":
        reader = PdfReader(str(src))
        pages = [
            page.extract_text() or ""
            for page in reader.pages
        ]
        text = "\n".join(pages)
        page_count = len(pages)

    elif src.suffix.lower() == ".txt":
        text = src.read_text(encoding="utf-8")
        page_count = None

    else:
        raise ValueError("Expected a PDF or TXT source")

    if not text.strip():
        raise ValueError(
            "No extractable text found. "
            "Scanned PDFs may require OCR."
        )

    return {
        "source": src.name,
        "characters": len(text),
        "pages": page_count
    }


# Step 3: Define document preparation
def prepare_documents(inputs, out, cfg):
    src = inputs[0]

    # Extract text
    if src.suffix.lower() == ".pdf":
        reader = PdfReader(str(src))
        text = "\n".join(
            page.extract_text() or ""
            for page in reader.pages
        )

    elif src.suffix.lower() == ".txt":
        text = src.read_text(encoding="utf-8")

    else:
        raise ValueError("Expected a PDF or TXT source")

    # Apply the same cleaning used in Part B
    text = re.sub(r"[ \t]+", " ", text).strip()

    size = int(cfg["chunk_size"])
    method = cfg["method"]

    if size <= 0:
        raise ValueError("Chunk size must be positive")

    # Fixed-size chunking
    if method == "fixed":
        chunks = [
            text[i:i + size]
            for i in range(0, len(text), size)
        ]

    # Paragraph-aware chunking
    elif method == "paragraph-aware":
        chunks = []
        current = ""

        for section in re.split(r"\n\s*\n", text):
            section = section.strip()

            if not section:
                continue

            if len(section) > size:
                if current:
                    chunks.append(current)
                    current = ""

                chunks.extend(
                    section[i:i + size]
                    for i in range(0, len(section), size)
                )

            elif len(current) + len(section) + (2 if current else 0) <= size:
                current += (
                    "\n\n" if current else ""
                ) + section

            else:
                if current:
                    chunks.append(current)

                current = section

        if current:
            chunks.append(current)

    else:
        raise ValueError(
            "Choose 'fixed' or 'paragraph-aware'"
        )

    if not chunks:
        raise ValueError("No text chunks produced")

    # Step 4: Store prepared document records
    target = out / "prepared_text.jsonl"

    with target.open("w", encoding="utf-8") as handle:
        for i, chunk in enumerate(chunks, 1):
            record = {
                "id": i,
                "source": str(src),
                "text": chunk,
                "page": None
            }
            handle.write(
                json.dumps(record, ensure_ascii=False) + "\n"
            )

    # Step 5: Return source-to-output mapping
    mapping = [
        {
            "source": str(src),
            "output": str(target)
        }
    ]

    processing_method = (
        f"{method} text chunks of {size} characters"
    )

    return [target], mapping, processing_method

# Step 6: Execute and register the pipeline
run_A = register_and_run(
    "documents/S05",
    SOURCE_A,
    inspect_documents,
    prepare_documents,
    CONFIG_A
)
print("Cell 8A completed successfully.")

SUCCESS: documents/S05 | Inputs: 1 | Outputs: 1
Manifest: C:\vscode\oya\ai-engineering-utm-academicmate\data\pipeline_v1\documents\S05\pipeline_manifest.json
Cell 8A completed successfully.


### Screenshot run and final checks

Cell 8C, pages 52–53, uses the same size as Part B. Cell 9, page 62, checks that every registered run gives the same source hashes, output hashes, sizes and links on the second pass.

In [37]:
# Cell 8C — CONDITIONAL REQUIRED: Computer Vision Projects
# Requires Cell 7 — Common V1 Pipeline Utilities

from PIL import Image

# Step 1: Select source folder and preparation settings
# Replace the dummy path with your actual image folder.

SOURCE_C = resolve_source(
    ".",
    folder=True
)

# Match the configuration selected in Part B.
CONFIG_C = {
    "resize": [1413, 837],
    "convert_rgb": True
}
# Step 2: Inspect source images
def inspect_images(inputs, cfg):
    properties = []
    for src in inputs:
        # Verify that the image can be opened
        with Image.open(src) as im:
            im.verify()
        # Reopen after verification to inspect properties
        with Image.open(src) as im:
            properties.append({
                "name": src.name,
                "size": list(im.size),
                "mode": im.mode
            })
    return {
        "image_count": len(inputs),
        "sample": properties[:3]
    }
# Step 3: Prepare images
def prepare_images(inputs, out, cfg):
    outputs = []
    mapping = []
    image_size = cfg["resize"]
    convert_rgb = cfg["convert_rgb"]
    for i, src in enumerate(inputs, 1):
        with Image.open(src) as im:
            # Apply selected colour conversion
            if convert_rgb:
                image = im.convert("RGB")
            else:
                image = im.copy()

            # Apply resizing only if selected in Part B
            if image_size is not None:
                image = image.resize(tuple(image_size))

            # Save prepared image
            target = out / f"prepared_{i:03d}.png"
            image.save(target)

        # Record output
        outputs.append(target)

        # Preserve source-to-output relationship
        mapping.append({
            "source": str(src),
            "output": str(target)
        })
    # Describe actual processing decisions
    processing_method = (
        f"convert_rgb={convert_rgb}; "
        f"resize={image_size}"
    )
    return outputs, mapping, processing_method
# Step 4: Execute and register the pipeline
run_C = register_and_run(
    "images",
    SOURCE_C,
    inspect_images,
    prepare_images,
    CONFIG_C
)
print("Cell 8C completed successfully.")

SUCCESS: images | Inputs: 2 | Outputs: 2
Manifest: C:\vscode\oya\ai-engineering-utm-academicmate\data\pipeline_v1\images\pipeline_manifest.json
Cell 8C completed successfully.


In [38]:
# Cell 9 — REQUIRED: Repeatability and Provenance Verification
# Run after your selected Cell 8A–8F pathway(s).

# Step 1: Ensure at least one pipeline has been registered
assert PIPELINE_RUNS, (
    "No pipeline runs registered. "
    "Run at least one selected Cell 8 pathway first."
)
print("--- Repeatability Verification ---")
# Step 2: Rerun each selected pipeline
for kind, entry in PIPELINE_RUNS.items():
    first = entry["first"]
    # Execute the same pipeline again
    second = entry["runner"]()
    # Step 3: Verify input provenance
    assert first["sources"] == second["sources"], (
        f"Source fingerprint changed for {kind}"
    )
    # Step 4: Verify prepared output fingerprints
    assert first["validation"] == second["validation"], (
        f"Prepared output changed for {kind}"
    )
    # Step 5: Verify source-to-output mapping
    assert (
        first["source_output_mapping"]
        == second["source_output_mapping"]
    ), (
        f"Source-to-output mapping changed for {kind}"
    )
    print(
        f"PASS: {kind} — source fingerprints, "
        "prepared outputs and mappings match"
    )
# Step 6: Display final verification message
print("\nAll selected pipelines passed repeatability smoke checks.")

--- Repeatability Verification ---
SUCCESS: documents/S01 | Inputs: 1 | Outputs: 1
Manifest: C:\vscode\oya\ai-engineering-utm-academicmate\data\pipeline_v1\documents\S01\pipeline_manifest.json
PASS: documents/S01 — source fingerprints, prepared outputs and mappings match
SUCCESS: documents | Inputs: 1 | Outputs: 1
Manifest: C:\vscode\oya\ai-engineering-utm-academicmate\data\pipeline_v1\documents\pipeline_manifest.json
PASS: documents — source fingerprints, prepared outputs and mappings match
SUCCESS: documents/S03 | Inputs: 1 | Outputs: 1
Manifest: C:\vscode\oya\ai-engineering-utm-academicmate\data\pipeline_v1\documents\S03\pipeline_manifest.json
PASS: documents/S03 — source fingerprints, prepared outputs and mappings match
SUCCESS: documents/S04 | Inputs: 1 | Outputs: 1
Manifest: C:\vscode\oya\ai-engineering-utm-academicmate\data\pipeline_v1\documents\S04\pipeline_manifest.json
PASS: documents/S04 — source fingerprints, prepared outputs and mappings match
SUCCESS: documents/S05 | Inpu

In Colab, use **File → Download → Download .ipynb** to keep the run. The Files panel contains the prepared outputs under **week3_project/data**. Save them before the runtime resets. Keep the complete course text and screenshots private.